### Feature Engineering

In [1]:
import pandas as pd
import numpy as np

##### Loading cleaned dataset

In [2]:
df = pd.read_csv("cleaned_data.csv")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397884 entries, 0 to 397883
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    397884 non-null  int64  
 1   StockCode    397884 non-null  object 
 2   Description  397884 non-null  object 
 3   Quantity     397884 non-null  int64  
 4   InvoiceDate  397884 non-null  object 
 5   UnitPrice    397884 non-null  float64
 6   CustomerID   397884 non-null  float64
 7   Country      397884 non-null  object 
 8   TotalPrice   397884 non-null  float64
dtypes: float64(3), int64(2), object(4)
memory usage: 27.3+ MB


In [4]:
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])

##### snapshot date

In [5]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

##### RFM features

In [6]:
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

##### Average order value

In [7]:
aov = df.groupby('CustomerID')['TotalPrice'].mean().reset_index()
aov.columns = ['CustomerID', 'AvgOrderValue']

##### Purchase frequency rate

In [8]:
purchase_freq = df.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
purchase_freq.columns = ['CustomerID', 'NumTransactions']

customer_days = df.groupby('CustomerID')['InvoiceDate'].agg(['min', 'max']).reset_index()
customer_days['CustomerLifespan'] = (customer_days['max'] - customer_days['min']).dt.days + 1

purchase_freq = purchase_freq.merge(customer_days[['CustomerID', 'CustomerLifespan']], on='CustomerID')

purchase_freq['PurchaseFrequency'] = purchase_freq['NumTransactions'] / purchase_freq['CustomerLifespan']

##### Average Days Between Purchases

In [9]:
df_sorted = df.sort_values(['CustomerID', 'InvoiceDate'])

df_sorted['PrevPurchase'] = df_sorted.groupby('CustomerID')['InvoiceDate'].shift(1)

df_sorted['DaysBetween'] = (df_sorted['InvoiceDate'] - df_sorted['PrevPurchase']).dt.days

avg_days_between = df_sorted.groupby('CustomerID')['DaysBetween'].mean().reset_index()
avg_days_between.columns = ['CustomerID', 'AvgDaysBetweenPurchases']

##### Merge All Features

In [11]:
features = rfm.merge(aov, on='CustomerID')
features = features.merge(purchase_freq[['CustomerID', 'PurchaseFrequency']], on='CustomerID')
features = features.merge(avg_days_between, on='CustomerID')

features.head()

,CustomerID,Recency,Frequency,Monetary,AvgOrderValue,PurchaseFrequency,AvgDaysBetweenPurchases
0,12346.0,326,1,77183.60,77183.600000,1.000000,NaN
1,12347.0,2,7,4310.00,23.681319,0.019126,2.0
2,12348.0,75,4,1797.24,57.975484,0.014134,9.4
3,12349.0,19,1,1757.55,24.076027,1.000000,0.0
4,12350.0,310,1,334.40,19.670588,1.000000,0.0


##### Create Target Variable

In [12]:
clv = df.groupby('CustomerID')['TotalPrice'].sum().reset_index()
clv.columns = ['CustomerID', 'CLV']

features = features.merge(clv, on='CustomerID')

In [13]:
features = features.fillna(0) #filling na rows

##### Save final dataset

In [14]:
features.to_csv("feature_data.csv", index=False)

#### Summary

- Created RFM features (Recency, Frequency, Monetary)
- Engineered advanced behavioral features:
  - Avg Order Value
  - Purchase Frequency
  - Avg Days Between Purchases
- Created CLV target variable
- Final dataset is ready for machine learning